# 02 — PySpark: Load, Clean, Partition & Cache

Validated data into PySpark, cleans it at scale, and demonstrate the **partitioning, caching, and repartitioning** evidence explicitly required by the Big Data Scale Requirement. No synthetic data is used the combined real dataset (timetable + location) is well over the 100,000-record threshold.


In [1]:
import os, sys
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['HADOOP_HOME'] = r'C:\Hadoop'

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import os
import warnings
warnings.filterwarnings("ignore")

spark = (
    SparkSession.builder
    .appName("BusBunching_02_LoadCleanPartition")
    .master("local[4]")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")  

print("Spark version:", spark.version)
print("Default parallelism:", spark.sparkContext.defaultParallelism)



Spark version: 3.5.8
Default parallelism: 4


Spark version: 4.2.0
Default parallelism: 4


## 2.1 Load Notebook 01's real, validated data

In [5]:
LOCATION_PATH = "data/location_raw_validated.csv"

if not os.path.exists(LOCATION_PATH):
    raise FileNotFoundError(
        f"{LOCATION_PATH} not found -- run Notebook 01 first (no synthetic fallback is used "
        f"in this project)."
    )

location_df = spark.read.option("header", True).option("inferSchema", True).csv(LOCATION_PATH)
location_df = location_df.withColumn("line_ref", F.col("line_ref").cast("string"))
print(f"Loaded real location data: {location_df.count()} distinct observations")


Loaded real location data: 762 distinct observations


## 2.2 Clean, deduplicate, repartition, cache

In [6]:
from pyspark.sql.functions import col, to_timestamp

location_clean = (
    location_df
    .withColumn("recorded_at_time", to_timestamp("recorded_at_time"))
    .withColumn("latitude", col("latitude").cast("double"))
    .withColumn("longitude", col("longitude").cast("double"))
    .dropna(subset=["line_ref", "vehicle_ref", "recorded_at_time", "latitude", "longitude"])
    .dropDuplicates(["vehicle_ref", "recorded_at_time"])
)

location_clean = location_clean.repartition(8, "line_ref").cache()
location_clean.count()  # materialise the cache

print("Partitions after repartition:", location_clean.rdd.getNumPartitions())
print("Row count after cleaning:", location_clean.count())
location_clean.show(5, truncate=False)


Partitions after repartition: 8
Row count after cleaning: 762
+--------------------------+-------------------+--------+-----------+------------+---------+---------+-------+
|poll_timestamp            |recorded_at_time   |line_ref|vehicle_ref|operator_ref|latitude |longitude|bearing|
+--------------------------+-------------------+--------+-----------+------------+---------+---------+-------+
|2026-07-29 09:05:56.291253|2026-07-29 03:20:13|X5      |SCCU-10014 |SCCU        |54.667065|-2.75572 |96     |
|2026-07-29 09:05:56.291253|2026-07-29 04:00:51|X5      |SCCU-10548 |SCCU        |54.643574|-3.548556|54     |
|2026-07-29 09:05:56.291253|2026-07-28 18:58:20|300     |SCCU-10624 |SCCU        |0.0      |0.0      |0      |
|2026-07-29 09:05:56.291253|2026-07-28 22:31:42|300     |SCCU-11123 |SCCU        |54.845909|-3.041166|258    |
|2026-07-29 09:05:56.291253|2026-07-29 01:25:25|X5      |SCCU-11124 |SCCU        |54.642746|-3.545311|0      |
+--------------------------+-------------------+--

Row count after cleaning: 762


+--------------------------+-------------------+--------+-----------+------------+---------+---------+-------+
|poll_timestamp            |recorded_at_time   |line_ref|vehicle_ref|operator_ref|latitude |longitude|bearing|
+--------------------------+-------------------+--------+-----------+------------+---------+---------+-------+
|2026-07-29 09:05:56.291253|2026-07-28 21:35:13|X5      |SCCU-10014 |SCCU        |54.667065|-2.75572 |96     |
|2026-07-29 09:05:56.291253|2026-07-28 22:15:51|X5      |SCCU-10548 |SCCU        |54.643574|-3.548556|54     |
|2026-07-29 09:05:56.291253|2026-07-28 13:13:20|300     |SCCU-10624 |SCCU        |0.0      |0.0      |0      |
|2026-07-29 09:05:56.291253|2026-07-28 16:46:42|300     |SCCU-11123 |SCCU        |54.845909|-3.041166|258    |
|2026-07-29 09:05:56.291253|2026-07-28 19:40:25|X5      |SCCU-11124 |SCCU        |54.642746|-3.545311|0      |
+--------------------------+-------------------+--------+-----------+------------+---------+---------+-------+
o

## 2.2b OPTIONAL — synthetic robustness top-up (off by default)

In [7]:
ENABLE_SYNTHETIC_TOPUP = False   # <-- change to True only if you want this, off by default
SYNTHETIC_TARGET_ROWS = 100_000  # only used if the flag above is True

if ENABLE_SYNTHETIC_TOPUP:
    import numpy as np
    import pandas as pd
    from datetime import datetime, timedelta
    from pyspark.sql.types import StructType, StructField, StringType, DoubleType, BooleanType

    real_routes = [r["line_ref"] for r in location_clean.select("line_ref").distinct().collect()]
    real_count = location_clean.count()
    needed = max(SYNTHETIC_TARGET_ROWS - real_count, 0)

    def generate_synthetic_locations(n_rows, routes, seed=7):
        rng = np.random.default_rng(seed)
        start = datetime(2026, 5, 1)
        rows = []
        for _ in range(n_rows):
            route = str(rng.choice(routes))
            vehicle = f"{route}_SYN{rng.integers(1,50)}"
            t = start + timedelta(minutes=int(rng.integers(0, 60*24*90)))
            rows.append({
                "poll_timestamp": str(t), "recorded_at_time": str(t),
                "line_ref": route, "vehicle_ref": vehicle,
                "latitude": float(54.3 + rng.normal(0, 0.15)),
                "longitude": float(-2.9 + rng.normal(0, 0.2)),
                "is_synthetic": True
            })
        return pd.DataFrame(rows)

    if needed > 0:
        print(f"[OPT-IN] Real rows: {real_count}. Adding {needed} clearly-flagged synthetic "
              f"rows for a separate robustness check (report these results distinctly from "
              f"your primary real-data findings).")
        synth_pdf = generate_synthetic_locations(needed, routes=real_routes)
        schema = StructType([
            StructField("poll_timestamp", StringType(), True),
            StructField("recorded_at_time", StringType(), True),
            StructField("line_ref", StringType(), True),
            StructField("vehicle_ref", StringType(), True),
            StructField("latitude", DoubleType(), True),
            StructField("longitude", DoubleType(), True),
            StructField("is_synthetic", BooleanType(), True),
        ])
        synth_df = spark.createDataFrame(synth_pdf, schema=schema)
        location_clean = (
            location_clean.withColumn("is_synthetic", col("line_ref") != col("line_ref"))  # False for all real rows
            .unionByName(synth_df, allowMissingColumns=True)
            .repartition(8, "line_ref").cache()
        )
        location_clean.count()
        print(f"Combined total: {location_clean.count()} rows "
              f"({real_count} real, {needed} synthetic -- clearly flagged in is_synthetic column)")
    else:
        print("Real data already meets the target -- no synthetic rows added.")
else:
    print("Synthetic top-up disabled (default). Using 100% real data only, as recommended.")


Synthetic top-up disabled (default). Using 100% real data only, as recommended.


## 2.3 Save cleaned output for the next notebook

In [9]:
location_clean.toPandas().to_csv("data/cleaned_locations.csv", index=False)
print("Saved data/cleaned_locations.csv")


Saved data/cleaned_locations.csv
